In [131]:
import numpy as np
import pandas as pd
import pyarrow

print("PyArrow:", pyarrow.__version__)
print("Pandas:", pd.__version__)

PyArrow: 25.0.1
Pandas: 3.0.5


In [132]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [133]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ealaxi/paysim1")

print("Path to dataset files:", path)

Path to dataset files: /Users/saidattaputta/.cache/kagglehub/datasets/ealaxi/paysim1/versions/2


In [134]:
import os

print(os.listdir(path))

['PS_20174392719_1491204439457_log.csv']


In [135]:
import os

file_path = os.path.join(
    path,
    'PS_20174392719_1491204439457_log.csv'
)

df = pd.read_csv(file_path)

df.head(5)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## Dataset Understanding
* One row represents One transaction/payment
* Time granularity is for every transaction/payment
* Columns that appear to be:
  * User identifier(s): nameOrig, nameDest
  * Monetary value(s): amount, oldbalanceOrg, newbalanceOrg, oldbalanceDest, newbalanceDest
  * Time variable(s): step
  * Outcome / status / risk signal(s): isFraud, isFlaggedFraud

In [136]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='str')

In [137]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [138]:
required_cols = ['step','type','amount','oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest','newbalanceDest','isFraud']

clean_df = df[required_cols].copy()
clean_df.head(5)

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0


In [139]:
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 8 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   oldbalanceOrg   float64
 4   newbalanceOrig  float64
 5   oldbalanceDest  float64
 6   newbalanceDest  float64
 7   isFraud         int64  
dtypes: float64(5), int64(2), str(1)
memory usage: 388.3 MB


In [140]:
clean_df.isna().sum()

step              0
type              0
amount            0
oldbalanceOrg     0
newbalanceOrig    0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
dtype: int64

No Null Values

In [141]:
clean_df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00


## Balance Consistensy

In [142]:
Origin_check = clean_df['oldbalanceOrg'] - clean_df['amount'] == clean_df['newbalanceOrig']

if Origin_check.all():
    print("Origin Balance is Equal")
else:
    print("Origin Balance is Not Equal")


Dest_check = clean_df['oldbalanceDest'] + clean_df['amount'] == clean_df['newbalanceDest']
if Dest_check.all():
    print("Destination Balance is Equal")
else:
    print("Destination Balance is Not Equal")


Origin Balance is Not Equal
Destination Balance is Not Equal


In [143]:
count_org = (df['oldbalanceOrg'] - df['amount'] != df['newbalanceOrig']).sum()
print("Number of rows where Origin Balance inconsistency is there:", count_org)

count_dest = (df['oldbalanceDest'] + df['amount'] != df['newbalanceDest']).sum()
print("Number of rows where Dest Balance inconsistency is there:", count_dest)

Number of rows where Origin Balance inconsistency is there: 5413997
Number of rows where Dest Balance inconsistency is there: 4736674


In [144]:
per_count_org = (count_org/6362620)*100
print("Percentage of rows where Origin Balance inconsisteny:",per_count_org)

per_count_dest = (count_dest/6362620)*100
print("Percentage of rows where dest Balance inconsisteny",per_count_dest)

Percentage of rows where Origin Balance inconsisteny: 85.09068591240715
Percentage of rows where dest Balance inconsisteny 74.44533855550071


In [145]:
origin_check = df['oldbalanceOrg'] - df['amount'] != df['newbalanceOrig']
fra_bal_check_org = (origin_check & (df['isFraud'] == 1)).sum()
print("Origin Balance inconsistency related to fraud:", fra_bal_check_org)

Dest_check = df['oldbalanceDest'] + df['amount'] != df['newbalanceDest']
fra_bal_check_dest = (Dest_check & (df['isFraud'] == 1)).sum()
print("Dest Balance inconsistency related to fraud:", fra_bal_check_dest)

Origin Balance inconsistency related to fraud: 127
Dest Balance inconsistency related to fraud: 5324


In [146]:
per_count_org_fra = (fra_bal_check_org/count_org)*100
print("Percentage of rows where Origin Balance inconsisteny related to fraud:",per_count_org_fra)

per_count_dest_fra = (fra_bal_check_dest/count_dest)*100
print("Percentage of rows where dest Balance inconsisteny related to fraud",per_count_dest_fra)

Percentage of rows where Origin Balance inconsisteny related to fraud: 0.002345771525178163
Percentage of rows where dest Balance inconsisteny related to fraud 0.11239954449050113


## Impossible Transactions

In [147]:
if (df['amount'] < 0).any():
    print("Negative amount exists")
else:
    print("None")
    
if (df['oldbalanceOrg'] < 0).any():
    print("Negative oldbalanceOrg exists")
else:
    print("None")
    
if (df['newbalanceOrig'] < 0).any():
    print("Negative newbalanceOrig exists")
else:
    print("None")
    
if (df['oldbalanceDest'] < 0).any():
    print("Negative oldbalanceDest exists")
else:
    print("None")
    
if (df['newbalanceDest'] < 0).any():
    print("Negative newbalanceDest exists")
else:
    print("None")

if(df['amount']>df['oldbalanceOrg']).any():
    print("amount greater than balance exist")
else:
    print("None")


None
None
None
None
None
amount greater than balance exist


In [148]:
count = (df['amount']>df['oldbalanceOrg']).sum()
print("Number of rows where amount > oldbalanceOrg:", count)

Number of rows where amount > oldbalanceOrg: 4079080


In [149]:
per_count = (count/6362620)*100
print("Percentage of rows where amount > oldbalanceOrg:",per_count)

Percentage of rows where amount > oldbalanceOrg: 64.11006786512475


## Duplicates

In [150]:
clean_df.duplicated().sum()

np.int64(543)

In [151]:
clean_df = clean_df.drop_duplicates()
clean_df.duplicated().sum()

np.int64(0)

## USER-TIME BALANCE VIEW

In [152]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='str')

In [153]:
required_cols_2 = ['nameOrig', 'step', 'amount', 'newbalanceOrig','isFraud']

user_balance_df = df[required_cols_2].copy()
user_balance_df.head(5)

,nameOrig,step,amount,newbalanceOrig,isFraud
0,C1231006815,1,9839.64,160296.36,0
1,C1666544295,1,1864.28,19384.72,0
2,C1305486145,1,181.00,0.00,1
3,C840083671,1,181.00,0.00,1
4,C2048537720,1,11668.14,29885.86,0


In [154]:
user_balance_df.duplicated().sum()

np.int64(0)

In [155]:
user_balance_df.info

<bound method DataFrame.info of             nameOrig  step      amount  newbalanceOrig  isFraud
0        C1231006815     1     9839.64       160296.36        0
1        C1666544295     1     1864.28        19384.72        0
2        C1305486145     1      181.00            0.00        1
3         C840083671     1      181.00            0.00        1
4        C2048537720     1    11668.14        29885.86        0
...              ...   ...         ...             ...      ...
6362615   C786484425   743   339682.13            0.00        1
6362616  C1529008245   743  6311409.28            0.00        1
6362617  C1162922333   743  6311409.28            0.00        1
6362618  C1685995037   743   850002.52            0.00        1
6362619  C1280323807   743   850002.52            0.00        1

[6362620 rows x 5 columns]>

In [156]:
user_balance_df = user_balance_df.sort_values(['nameOrig','step'])

In [157]:
unique_users = user_balance_df['nameOrig'].nunique()
total_rows = len(user_balance_df)
per_uni_userid = (unique_users/total_rows)*100
print("Unique No.of users:", per_uni_userid)

Unique No.of users: 99.85362947967975


In [158]:
print("Total transactions:", total_rows)
print("Unique users:", unique_users)
print("Average transactions per user:", total_rows / unique_users)

Total transactions: 6362620
Unique users: 6353307
Average transactions per user: 1.0014658507766112


## Reconstruction of DataFrame

In [159]:
required_col_3 = ['nameOrig','nameDest','newbalanceDest', 'step', 'amount', 'newbalanceOrig','isFraud']

user_balance_df_3 = df[required_col_3].copy()
user_balance_df_3.head(5)

,nameOrig,nameDest,newbalanceDest,step,amount,newbalanceOrig,isFraud
0,C1231006815,M1979787155,0.0,1,9839.64,160296.36,0
1,C1666544295,M2044282225,0.0,1,1864.28,19384.72,0
2,C1305486145,C553264065,0.0,1,181.00,0.00,1
3,C840083671,C38997010,0.0,1,181.00,0.00,1
4,C2048537720,M1230701703,0.0,1,11668.14,29885.86,0


In [160]:
user_balance_df_3.duplicated().sum()

np.int64(0)

In [161]:
unique_users_dest = user_balance_df_3['nameDest'].nunique()
total_rows_2 = len(user_balance_df_3)
per_uni_dest_userid = (unique_users_dest/total_rows_2)*100
print("Unique No.of destination users:", per_uni_dest_userid)

Unique No.of destination users: 42.78680795018404


In [162]:
print("Total transactions:", total_rows_2)
print("Unique destination users:", unique_users_dest)
print("Average transactions per destination user:", total_rows_2 / unique_users_dest)

Total transactions: 6362620
Unique destination users: 2722362
Average transactions per destination user: 2.337168973119666


In [163]:
orig_user = set(user_balance_df_3['nameOrig'].unique())
dest_user = set(user_balance_df_3['nameDest'].unique())

only_orig = orig_user - dest_user
only_dest = dest_user - orig_user
both_roles = orig_user & dest_user

print("Only Orig:",len(only_orig))
print("Only dest:",len(only_dest))
print("Both roles:",len(both_roles))

Only Orig: 6351538
Only dest: 2720593
Both roles: 1769


In [164]:
orig_counts = user_balance_df_3['nameOrig'].value_counts()
dest_counts = user_balance_df_3['nameDest'].value_counts()

print("orig count:",orig_counts)
print("dest count:",dest_counts)

orig count: nameOrig
C2098525306    3
C400299098     3
C1999539787    3
C1065307291    3
C545315117     3
              ..
C786484425     1
C1529008245    1
C1162922333    1
C1685995037    1
C1280323807    1
Name: count, Length: 6353307, dtype: int64
dest count: nameDest
C1286084959    113
C985934102     109
C665576141     105
C2083562754    102
C248609774     101
              ... 
C1812552860      1
C1470998563      1
C1850423904      1
C1881841831      1
C2080388513      1
Name: count, Length: 2722362, dtype: int64


In [165]:
avg_only_orig = orig_counts[orig_counts.index.isin(only_orig)].mean()
avg_only_dest = dest_counts[dest_counts.index.isin(only_dest)].mean()

both_orig = orig_counts[orig_counts.index.isin(both_roles)]
both_dest = dest_counts[dest_counts.index.isin(both_roles)]

both_total = both_orig.add(both_dest, fill_value=0)
avg_both = both_total.mean()

print("Average transactions per user (Only Origin):", avg_only_orig)
print("Average transactions per user (Only Destination):", avg_only_dest)
print("Average transactions per user (Both Roles):", avg_both)

Average transactions per user (Only Origin): 1.0014659441539986
Average transactions per user (Only Destination): 2.3337970802688974
Average transactions per user (Both Roles): 8.524024872809497


## Unified User-Time Exposure Dataset

In [166]:
orig_df = user_balance_df_3[['nameOrig','step','amount','newbalanceOrig','isFraud']].copy()
orig_df.columns = ['user_id','step','amount','balance_after','isFraud']
orig_df['role'] = 'origin'
orig_df['amount'] = -orig_df['amount']

desf_df = user_balance_df_3[['nameDest','step','amount','newbalanceDest','isFraud']].copy()
desf_df.columns = ['user_id','step','amount','balance_after','isFraud']
desf_df['role'] = 'dest'
desf_df['amount'] = +desf_df['amount']

unified_df = pd.concat([orig_df,desf_df],ignore_index = True)

unified_df = unified_df.sort_values(['user_id', 'step']).reset_index(drop=True)

### Data Validation

In [167]:
sample_users = unified_df['user_id'].drop_duplicates().sample(5,random_state = 42)
sample_df = unified_df[unified_df['user_id'].isin(sample_users)]

In [168]:
def step_monotonically(user_df):
    return user_df['step'].is_monotonic_increasing
def step_decreasing(user_df):
    return (user_df['step'].diff()<0).any()

step_check = sample_df.groupby('user_id').apply(step_monotonically)
print("Step monotonicity check:")
print(step_check)

step_decreasing_check = sample_df.groupby('user_id').apply(step_decreasing)
print("Users with decreasing steps:")
print(step_decreasing_check)

Step monotonicity check:
user_id
C1166016147    True
C1682102083    True
C186406807     True
C66247056      True
C990778067     True
dtype: bool
Users with decreasing steps:
user_id
C1166016147    False
C1682102083    False
C186406807     False
C66247056      False
C990778067     False
dtype: bool


In [169]:
def detect_large_balance_jumps(user_df):
    return (user_df['balance_after'].diff().abs() > user_df['balance_after'].std()*5).any()

oscillation_check = sample_df.groupby('user_id').apply(detect_large_balance_jumps)
print("Users with suspicious balance oscillations:")
print(oscillation_check)

Users with suspicious balance oscillations:
user_id
C1166016147    False
C1682102083    False
C186406807     False
C66247056      False
C990778067     False
dtype: bool


In [170]:
total_rows_unified = len(unified_df)
unique_users_unified = unified_df['user_id'].nunique()
avg_tx_unified = total_rows_unified / unique_users_unified

print("Unified total rows:", total_rows_unified)
print("Unified unique users:", unique_users_unified)
print("Avg transactions per unified user:", avg_tx_unified)


Unified total rows: 12725240
Unified unique users: 9073900
Avg transactions per unified user: 1.402400290944357


In [171]:
unified_df['balance_after'] = unified_df['balance_after'].astype('float32')
unified_df['amount'] = unified_df['amount'].astype('float32')

In [172]:
del df
import gc
gc.collect()

118

## Implementation

### Implementing Rolling Peak

In [173]:
unified_df['rolling_peak_3'] = (
    unified_df
    .groupby('user_id')['balance_after']
    .rolling(window=3, min_periods=1)
    .max()
    .reset_index(level=0, drop=True)
)

In [174]:
duplicate_user_steps = (
    unified_df
    .groupby(['user_id', 'step'])
    .size()
    .reset_index(name='count')
)

print(
    "User-step combinations with multiple records:",
    (duplicate_user_steps['count'] > 1).sum()
)

print(
    "Maximum records within one user-step:",
    duplicate_user_steps['count'].max()
)

User-step combinations with multiple records: 249241
Maximum records within one user-step: 32


### Implement Drawdown

In [175]:
unified_df['drawdown'] = np.where(
    unified_df['rolling_peak_3'] > 0,
    (
        unified_df['rolling_peak_3']
        - unified_df['balance_after']
    ) / unified_df['rolling_peak_3'],
    0
)

In [176]:
unified_df['rolling_peak_3'].describe()

count    1.272524e+07
mean     1.061233e+06
std      3.332702e+06
min      0.000000e+00
25%      0.000000e+00
50%      1.966569e+04
75%      6.159766e+05
max      3.561793e+08
Name: rolling_peak_3, dtype: float64

In [177]:
unified_df.to_pickle("unified_user_exposure.pkl")

print("Unified DataFrame saved successfully")

Unified DataFrame saved successfully


In [178]:
# Fast reload for future notebook sessions

import pandas as pd

unified_df = pd.read_pickle("unified_user_exposure.pkl")

print("Unified DataFrame loaded successfully")
print("Shape:", unified_df.shape)

Unified DataFrame loaded successfully
Shape: (12725240, 8)


In [179]:
# Dataset checkpoint saved using Pickle for fast local reload.
# unified_df.to_parquet(...) is intentionally not used in this local workflow.

In [180]:
sample_users = unified_df['user_id'].drop_duplicates().sample(
    300000,
    random_state=42
)

dev_df = unified_df[
    unified_df['user_id'].isin(sample_users)
].copy()

dev_df.to_pickle("unified_dev_sample.pkl")

print("Development sample saved successfully")
print("Shape:", dev_df.shape)

Development sample saved successfully
Shape: (418269, 8)


In [181]:
dev_df = pd.read_pickle("unified_dev_sample.pkl")

print("Development sample loaded successfully")
print("Shape:", dev_df.shape)

Development sample loaded successfully
Shape: (418269, 8)


In [182]:
# Check duplicate user-step combinations in the development sample

duplicate_dev_steps = (
    dev_df
    .groupby(['user_id', 'step'])
    .size()
    .reset_index(name='transaction_count')
)

duplicate_cases = duplicate_dev_steps[
    duplicate_dev_steps['transaction_count'] > 1
]

print("Duplicate user-step combinations:", len(duplicate_cases))
print(
    "Maximum transactions within one user-step:",
    duplicate_dev_steps['transaction_count'].max()
)

duplicate_cases.head(10)

Duplicate user-step combinations: 8018
Maximum transactions within one user-step: 22


,user_id,step,transaction_count
61,C1000545031,522,2
76,C1000621564,19,3
77,C1000621564,21,2
93,C1000621564,187,2
137,C1000848987,155,2
205,C1001114349,134,2
227,C1001181764,162,2
477,C1002407360,353,2
497,C1002475798,329,2
506,C1002497793,129,2


In [183]:
print("Columns:")
print(dev_df.columns.tolist())

print("\nData types:")
print(dev_df.dtypes)

print("\nFirst 5 rows:")
display(dev_df.head())

Columns:
['user_id', 'step', 'amount', 'balance_after', 'isFraud', 'role', 'rolling_peak_3', 'drawdown']

Data types:
user_id               str
step                int64
amount            float32
balance_after     float32
isFraud             int64
role                  str
rolling_peak_3    float64
drawdown          float64
dtype: object

First 5 rows:


,user_id,step,amount,balance_after,isFraud,role,rolling_peak_3,drawdown
2,C1000001725,46,-8424.740234,0.0,0,origin,0.0,0.0
95,C1000018372,156,-108365.468750,18449576.0,0,origin,18449576.0,0.0
128,C1000022742,18,-2368.679932,0.0,0,origin,0.0,0.0
379,C1000069522,329,-590353.812500,13756703.0,0,origin,13756703.0,0.0
489,C1000089317,140,-283090.375000,0.0,0,origin,0.0,0.0


In [184]:
display(dev_df.head())

,user_id,step,amount,balance_after,isFraud,role,rolling_peak_3,drawdown
2,C1000001725,46,-8424.740234,0.0,0,origin,0.0,0.0
95,C1000018372,156,-108365.468750,18449576.0,0,origin,18449576.0,0.0
128,C1000022742,18,-2368.679932,0.0,0,origin,0.0,0.0
379,C1000069522,329,-590353.812500,13756703.0,0,origin,13756703.0,0.0
489,C1000089317,140,-283090.375000,0.0,0,origin,0.0,0.0


In [185]:
# Inspect rolling_peak_3 distribution and role values

print("Role distribution:")
print(dev_df["role"].value_counts())

print("\nrolling_peak_3 summary:")
print(dev_df["rolling_peak_3"].describe())

print("\nRows where rolling_peak_3 is greater than 0:")
display(
    dev_df.loc[
        dev_df["rolling_peak_3"] > 0,
        ["user_id", "step", "amount", "balance_after", "role", "rolling_peak_3"]
    ].head(10)
)

Role distribution:
role
origin    210524
dest      207745
Name: count, dtype: int64

rolling_peak_3 summary:
count    4.182690e+05
mean     1.056983e+06
std      3.120438e+06
min      0.000000e+00
25%      0.000000e+00
50%      1.756421e+04
75%      6.046629e+05
max      2.059364e+08
Name: rolling_peak_3, dtype: float64

Rows where rolling_peak_3 is greater than 0:


,user_id,step,amount,balance_after,role,rolling_peak_3
95,C1000018372,156,-108365.468750,1.844958e+07,origin,1.844958e+07
379,C1000069522,329,-590353.812500,1.375670e+07,origin,1.375670e+07
533,C1000094896,184,-122524.546875,6.478340e+06,origin,6.478340e+06
663,C1000113762,229,-17855.490234,3.343655e+05,origin,3.343655e+05
1210,C1000210704,131,-128872.476562,1.716247e+07,origin,1.716247e+07
1225,C1000215475,262,-5786.770020,4.546337e+05,origin,4.546337e+05
1313,C1000231400,259,-9294.080078,1.832920e+03,origin,1.832920e+03
1446,C1000249904,298,213538.468750,2.493514e+05,dest,2.493514e+05
1447,C1000249904,307,66757.718750,1.825937e+05,dest,2.493514e+05
1512,C1000265437,347,-284308.875000,2.914269e+05,origin,2.914269e+05


In [186]:
print("Balance_after summary:")
print(dev_df["balance_after"].describe())

print("\nRolling peak summary:")
print(dev_df["rolling_peak_3"].describe())

print("\nCheck whether rolling_peak_3 can be a direct rolling maximum:")

comparison = dev_df[
    [
        "user_id",
        "step",
        "balance_after",
        "rolling_peak_3",
        "role"
    ]
].copy()

print(
    "Rows where rolling_peak_3 > balance_after:",
    (comparison["rolling_peak_3"] > comparison["balance_after"]).sum()
)

display(
    comparison[
        comparison["rolling_peak_3"] > 0
    ].head(15)
)

Balance_after summary:
count    4.182690e+05
mean     1.036228e+06
std      3.112989e+06
min      0.000000e+00
25%      0.000000e+00
50%      1.166264e+04
75%      5.507579e+05
max      2.059364e+08
Name: balance_after, dtype: float64

Rolling peak summary:
count    4.182690e+05
mean     1.056983e+06
std      3.120438e+06
min      0.000000e+00
25%      0.000000e+00
50%      1.756421e+04
75%      6.046629e+05
max      2.059364e+08
Name: rolling_peak_3, dtype: float64

Check whether rolling_peak_3 can be a direct rolling maximum:
Rows where rolling_peak_3 > balance_after: 43930


,user_id,step,balance_after,rolling_peak_3,role
95,C1000018372,156,1.844958e+07,1.844958e+07,origin
379,C1000069522,329,1.375670e+07,1.375670e+07,origin
533,C1000094896,184,6.478340e+06,6.478340e+06,origin
663,C1000113762,229,3.343655e+05,3.343655e+05,origin
1210,C1000210704,131,1.716247e+07,1.716247e+07,origin
1225,C1000215475,262,4.546337e+05,4.546337e+05,origin
1313,C1000231400,259,1.832920e+03,1.832920e+03,origin
1446,C1000249904,298,2.493514e+05,2.493514e+05,dest
1447,C1000249904,307,1.825937e+05,2.493514e+05,dest
1512,C1000265437,347,2.914269e+05,2.914269e+05,origin


In [188]:
# Inspect duplicate user-step records in detail

duplicate_user_steps = (
    dev_df
    .groupby(['user_id', 'step'])
    .filter(lambda x: len(x) > 1)
    .sort_values(['user_id', 'step'])
)

print("Total rows belonging to duplicate user-step groups:")
print(len(duplicate_user_steps))

print("\nSample duplicate user-step groups:")

sample_duplicate_groups = (
    duplicate_user_steps
    .groupby(['user_id', 'step'])
    .head(10)
)

display(
    sample_duplicate_groups[
        [
            'user_id',
            'step',
            'amount',
            'balance_after',
            'isFraud',
            'role',
            'rolling_peak_3',
            'drawdown'
        ]
    ].head(30)
)

Total rows belonging to duplicate user-step groups:
18239

Sample duplicate user-step groups:


,user_id,step,amount,balance_after,isFraud,role,rolling_peak_3,drawdown
3103,C1000545031,522,9750.259766,2.899301e+05,0,dest,2.899301e+05,0.000000
3104,C1000545031,522,188764.734375,4.786948e+05,0,dest,4.786948e+05,0.000000
3575,C1000621564,19,116089.171875,3.181545e+05,0,dest,3.181545e+05,0.000000
3576,C1000621564,19,118249.500000,3.473433e+04,0,dest,3.181545e+05,0.890826
3577,C1000621564,19,165170.687500,3.473433e+04,0,dest,3.181545e+05,0.890826
3578,C1000621564,21,198401.265625,2.331356e+05,0,dest,2.331356e+05,0.000000
3579,C1000621564,21,18793.050781,2.519286e+05,0,dest,2.519286e+05,0.000000
3595,C1000621564,187,78458.890625,1.808471e+06,0,dest,1.808471e+06,0.000000
3596,C1000621564,187,66707.726562,1.808471e+06,0,dest,1.808471e+06,0.000000
5006,C1000848987,155,226445.453125,1.060941e+06,0,dest,1.060941e+06,0.000000


In [189]:
# Recalculate rolling maximum independently using the CURRENT row order

dev_df["expected_rolling_peak"] = (
    dev_df
    .groupby("user_id")["balance_after"]
    .rolling(window=3, min_periods=1)
    .max()
    .reset_index(level=0, drop=True)
)

# Compare existing rolling_peak_3 with expected result

difference_count = (
    ~np.isclose(
        dev_df["rolling_peak_3"],
        dev_df["expected_rolling_peak"],
        equal_nan=True
    )
).sum()

print("Rows where rolling_peak_3 differs from expected:", difference_count)

# Show mismatches
mismatches = dev_df.loc[
    ~np.isclose(
        dev_df["rolling_peak_3"],
        dev_df["expected_rolling_peak"],
        equal_nan=True
    ),
    [
        "user_id",
        "step",
        "amount",
        "balance_after",
        "rolling_peak_3",
        "expected_rolling_peak",
        "role"
    ]
]

display(mismatches.head(20))

Rows where rolling_peak_3 differs from expected: 0


,user_id,step,amount,balance_after,rolling_peak_3,expected_rolling_peak,role


In [190]:
# Check index alignment and compare values directly

print("dev_df index unique:", dev_df.index.is_unique)
print("dev_df index sorted:", dev_df.index.is_monotonic_increasing)

print("\nCurrent balance_after range:")
print(dev_df["balance_after"].min(), "to", dev_df["balance_after"].max())

print("\nCurrent rolling_peak_3 range:")
print(dev_df["rolling_peak_3"].min(), "to", dev_df["rolling_peak_3"].max())

print("\nExpected rolling peak range:")
print(
    dev_df["expected_rolling_peak"].min(),
    "to",
    dev_df["expected_rolling_peak"].max()
)

# Show rows where rolling peak is non-zero
display(
    dev_df.loc[
        dev_df["rolling_peak_3"] > 0,
        [
            "user_id",
            "step",
            "balance_after",
            "rolling_peak_3",
            "expected_rolling_peak"
        ]
    ].head(20)
)

dev_df index unique: True
dev_df index sorted: True

Current balance_after range:
0.0 to 2.0593637e+08

Current rolling_peak_3 range:
0.0 to 205936368.0

Expected rolling peak range:
0.0 to 205936368.0


,user_id,step,balance_after,rolling_peak_3,expected_rolling_peak
95,C1000018372,156,1.844958e+07,1.844958e+07,1.844958e+07
379,C1000069522,329,1.375670e+07,1.375670e+07,1.375670e+07
533,C1000094896,184,6.478340e+06,6.478340e+06,6.478340e+06
663,C1000113762,229,3.343655e+05,3.343655e+05,3.343655e+05
1210,C1000210704,131,1.716247e+07,1.716247e+07,1.716247e+07
1225,C1000215475,262,4.546337e+05,4.546337e+05,4.546337e+05
1313,C1000231400,259,1.832920e+03,1.832920e+03,1.832920e+03
1446,C1000249904,298,2.493514e+05,2.493514e+05,2.493514e+05
1447,C1000249904,307,1.825937e+05,2.493514e+05,2.493514e+05
1512,C1000265437,347,2.914269e+05,2.914269e+05,2.914269e+05


In [191]:
# Check whether step values are monotonically increasing within each user

step_order_check = (
    dev_df
    .groupby("user_id")["step"]
    .apply(lambda x: x.is_monotonic_increasing)
)

print(
    "Users with steps in increasing order:",
    step_order_check.sum()
)

print(
    "Users with steps NOT in increasing order:",
    (~step_order_check).sum()
)

# Show example users where ordering is not monotonic
bad_users = step_order_check[
    ~step_order_check
].index.tolist()

print("\nSample users with non-monotonic steps:")
print(bad_users[:10])

Users with steps in increasing order: 300000
Users with steps NOT in increasing order: 0

Sample users with non-monotonic steps:
[]


In [192]:
# Find the drawdown column name first
print(dev_df.columns.tolist())

['user_id', 'step', 'amount', 'balance_after', 'isFraud', 'role', 'rolling_peak_3', 'drawdown', 'expected_rolling_peak']


In [193]:
# Check how drawdown relates to rolling_peak_3 and balance_after

dev_df[
    [
        "user_id",
        "step",
        "balance_after",
        "rolling_peak_3",
        "drawdown"
    ]
].head(20)

,user_id,step,balance_after,rolling_peak_3,drawdown
2,C1000001725,46,0.000000e+00,0.000000e+00,0.0
95,C1000018372,156,1.844958e+07,1.844958e+07,0.0
128,C1000022742,18,0.000000e+00,0.000000e+00,0.0
379,C1000069522,329,1.375670e+07,1.375670e+07,0.0
489,C1000089317,140,0.000000e+00,0.000000e+00,0.0
510,C1000093355,381,0.000000e+00,0.000000e+00,0.0
533,C1000094896,184,6.478340e+06,6.478340e+06,0.0
597,C1000104995,394,0.000000e+00,0.000000e+00,0.0
663,C1000113762,229,3.343655e+05,3.343655e+05,0.0
684,C1000118769,230,0.000000e+00,0.000000e+00,0.0


In [194]:
# Test absolute drawdown hypothesis
dev_df["expected_drawdown_abs"] = (
    dev_df["rolling_peak_3"] - dev_df["balance_after"]
)

# Test percentage drawdown hypothesis
dev_df["expected_drawdown_pct"] = (
    (dev_df["rolling_peak_3"] - dev_df["balance_after"])
    / dev_df["rolling_peak_3"].replace(0, float("nan"))
)

print(
    dev_df[
        [
            "drawdown",
            "expected_drawdown_abs",
            "expected_drawdown_pct"
        ]
    ].corr()
)

display(
    dev_df[
        [
            "user_id",
            "step",
            "balance_after",
            "rolling_peak_3",
            "drawdown",
            "expected_drawdown_abs",
            "expected_drawdown_pct"
        ]
    ].head(20)
)

                       drawdown  expected_drawdown_abs  expected_drawdown_pct
drawdown               1.000000               0.428362               1.000000
expected_drawdown_abs  0.428362               1.000000               0.409025
expected_drawdown_pct  1.000000               0.409025               1.000000


,user_id,step,balance_after,rolling_peak_3,drawdown,expected_drawdown_abs,expected_drawdown_pct
2,C1000001725,46,0.000000e+00,0.000000e+00,0.0,0.0,NaN
95,C1000018372,156,1.844958e+07,1.844958e+07,0.0,0.0,0.0
128,C1000022742,18,0.000000e+00,0.000000e+00,0.0,0.0,NaN
379,C1000069522,329,1.375670e+07,1.375670e+07,0.0,0.0,0.0
489,C1000089317,140,0.000000e+00,0.000000e+00,0.0,0.0,NaN
510,C1000093355,381,0.000000e+00,0.000000e+00,0.0,0.0,NaN
533,C1000094896,184,6.478340e+06,6.478340e+06,0.0,0.0,0.0
597,C1000104995,394,0.000000e+00,0.000000e+00,0.0,0.0,NaN
663,C1000113762,229,3.343655e+05,3.343655e+05,0.0,0.0,0.0
684,C1000118769,230,0.000000e+00,0.000000e+00,0.0,0.0,NaN


In [195]:
import numpy as np

# Compare actual drawdown with expected percentage drawdown
drawdown_diff = (
    dev_df["drawdown"] -
    dev_df["expected_drawdown_pct"]
)

print("Maximum absolute difference:",
      drawdown_diff.abs().max())

print("Rows with differences:",
      (~np.isclose(
          dev_df["drawdown"],
          dev_df["expected_drawdown_pct"],
          equal_nan=True
      )).sum())

Maximum absolute difference: 0.0
Rows with differences: 196540


In [196]:
# Investigate the mismatches

mismatch_mask = ~np.isclose(
    dev_df["drawdown"],
    dev_df["expected_drawdown_pct"],
    equal_nan=True
)

print("Total mismatches:", mismatch_mask.sum())

print("\nMismatch breakdown:")
print(
    dev_df.loc[mismatch_mask]
    .assign(
        peak_is_zero=lambda x: x["rolling_peak_3"] == 0,
        balance_is_zero=lambda x: x["balance_after"] == 0
    )[["rolling_peak_3", "balance_after", "drawdown",
       "expected_drawdown_pct", "peak_is_zero", "balance_is_zero"]]
    .value_counts(["peak_is_zero", "balance_is_zero"])
)

print("\nSample mismatches:")
display(
    dev_df.loc[
        mismatch_mask,
        [
            "user_id",
            "step",
            "balance_after",
            "rolling_peak_3",
            "drawdown",
            "expected_drawdown_pct"
        ]
    ].head(20)
)

Total mismatches: 196540

Mismatch breakdown:
peak_is_zero  balance_is_zero
True          True               196540
Name: count, dtype: int64

Sample mismatches:


,user_id,step,balance_after,rolling_peak_3,drawdown,expected_drawdown_pct
2,C1000001725,46,0.0,0.0,0.0,NaN
128,C1000022742,18,0.0,0.0,0.0,NaN
489,C1000089317,140,0.0,0.0,0.0,NaN
510,C1000093355,381,0.0,0.0,0.0,NaN
597,C1000104995,394,0.0,0.0,0.0,NaN
684,C1000118769,230,0.0,0.0,0.0,NaN
717,C1000124896,404,0.0,0.0,0.0,NaN
786,C1000135942,165,0.0,0.0,0.0,NaN
1002,C1000167233,404,0.0,0.0,0.0,NaN
1133,C1000191019,231,0.0,0.0,0.0,NaN


In [197]:
# Verify the complete drawdown business rule

dev_df["expected_drawdown_final"] = np.where(
    dev_df["rolling_peak_3"] == 0,
    0,
    (
        dev_df["rolling_peak_3"] - dev_df["balance_after"]
    ) / dev_df["rolling_peak_3"]
)

final_diff = (
    dev_df["drawdown"] -
    dev_df["expected_drawdown_final"]
)

print("Maximum absolute difference:",
      final_diff.abs().max())

print(
    "Rows with differences:",
    (~np.isclose(
        dev_df["drawdown"],
        dev_df["expected_drawdown_final"],
        equal_nan=True
    )).sum()
)

Maximum absolute difference: 0.0
Rows with differences: 0


In [199]:
dev_df["expected_drawdown_final"] = np.where(
    dev_df["rolling_peak_3"] == 0,
    0,
    (dev_df["rolling_peak_3"] - dev_df["balance_after"])
    / dev_df["rolling_peak_3"]
)

final_diff = (
    dev_df["drawdown"] -
    dev_df["expected_drawdown_final"]
)

print(
    "Maximum absolute difference:",
    final_diff.abs().max()
)

print(
    "Rows with differences:",
    (~np.isclose(
        dev_df["drawdown"],
        dev_df["expected_drawdown_final"],
        equal_nan=True
    )).sum()
)

Maximum absolute difference: 0.0
Rows with differences: 0


In [187]:
for file in os.listdir('.'):
    if file.startswith('unified_'):
        print(file, "-", round(os.path.getsize(file) / (1024 * 1024), 2), "MB")

unified_user_exposure.pkl - 705.52 MB
unified_dev_sample.pkl - 26.38 MB
